# Phase 2.1 : Pipeline d'augmentation

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase1_4_training.ipynb`.

**Ce notebook est autonome** : la section "Reprise" ci-dessous refait le setup des phases 1.1 à 1.2 (téléchargement, organisation, pipeline de données), sans les explications déjà vues.

## Reprise (setup phases 1.1 et 1.2, condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


@tf.function
def _try_decode(img_bytes):
    # Force l'execution en mode graphe (via tf.function), qui est plus stricte que
    # l'execution eager sur certains formats (ex. BMP mal forme deguise en .jpg) et
    # reproduit exactement le mode d'execution utilise par image_dataset_from_directory
    # pendant l'entrainement.
    return tf.io.decode_image(img_bytes, channels=3, expand_animations=False)


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ni tf.io.decode_image() en
    # mode eager ne suffisent : ce dataset contient des fichiers que seul le decodeur en
    # mode graphe rejette (InvalidArgumentError, "Unknown image file format" ou erreurs
    # de dimension BMP). On teste donc chaque fichier via une tf.function.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            _try_decode(img_bytes)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Phase 2.1 : Pipeline d'augmentation

**Objectif** : intégrer les couches d'augmentation Keras dans le modèle, vérifier visuellement les transformations sur une grille d'images augmentées avant l'entraînement.

Rappel : `RandomFlip`/`RandomRotation`/`RandomZoom` sont actives uniquement en mode `training=True` (donc pendant `model.fit`) et passives en inférence — la `val_accuracy` restera calculée sur les images originales, jamais augmentées.

In [ ]:
from tensorflow.keras import layers, models

# Ces couches sont actives uniquement en mode training (model.fit) et
# passives en mode inference (model.predict, model.evaluate).
# Consequence : la val_accuracy est calculee sur les images ORIGINALES. C'est voulu.
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),   # +/-10% de 360 deg = +/-36 deg
    layers.RandomZoom(0.1),       # +/-10% de zoom
], name="data_augmentation")

In [ ]:
import matplotlib.pyplot as plt

sample_image, _ = next(iter(train_ds))
sample_image = sample_image[0]

plt.figure(figsize=(9, 9))
for i in range(9):
    augmented = data_augmentation(tf.expand_dims(sample_image, 0), training=True)
    plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0])
    plt.axis("off")

plt.tight_layout()
plt.savefig("augmentation_grid.png", dpi=100)
plt.show()

### Checkpoint avant la phase 2.2

- **Happy path** : 9 versions augmentées de la même image affichées, toutes différentes (flip, rotation, zoom visibles), toutes reconnaissables comme appartenant à la même classe.
- **Edge case** : empiler 5x `RandomRotation(0.5)` d'affilée dans le pipeline rendrait l'image méconnaissable — plus n'est pas mieux. Si une transformation te semble ambiguë même visuellement, réduis son facteur.
- **Adversarial** : sur des photos naturelles (chats/chiens), `RandomFlip("vertical")` serait contre-productif — un animal la tête en bas n'existe pas dans le test set, le modèle apprendrait une invariance qui n'a pas de sens. On utilise donc uniquement `"horizontal"`.

**Prochaine étape** : Phase 2.2 — intégrer cette augmentation dans un CNN avec Dropout, et réentraîner depuis zéro.